# Euer eigenes RAG

Arbeitsnotebook zu [Anforderungen.md](Anforderungen.md). Begriffe stehen im [Glossar](../README.md#glossar).

**Thema:** ...

- Stellen mit `TODO` ergänzt ihr selbst.
- Zellen **der Reihe nach** ausführen. Nach einem Kernel-Neustart wieder oben anfangen.

# Imports

In [ ]:
import os
import re
import shutil
from pathlib import Path

# Anfragen an Ollama auf dem eigenen Rechner sollen nicht über einen Proxy laufen
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
# Kennung, mit der Websites abgerufen werden (sonst gibt es eine Warnung)
os.environ["USER_AGENT"] = "ausbildung-uebungen"

from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Phase 0: Erste Schritte mit Ollama

### Das LLM ansprechen

`temperature=0` sorgt dafür, dass gleiche Fragen möglichst gleiche Antworten bekommen. Das macht das Testen leichter.

In [ ]:
llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url="http://localhost:11434",
    temperature=0,
)

antwort = llm.invoke("Erkläre in drei Sätzen, was ein Large Language Model ist.")
print(antwort.content)

### Übung: Was weiß das LLM nicht?

Stellt eine Frage zu etwas, das nur in euren Dokumenten steht, z. B. zu eurem selbst erstellten PDF. Was antwortet das LLM?

In [ ]:
# TODO: Frage einsetzen
print(llm.invoke("...").content)

### Das Embedding-Modell ausprobieren

Ein Embedding macht aus einem Text eine Liste von Zahlen.

In [ ]:
embeddings = OllamaEmbeddings(model="embeddinggemma:300m", base_url="http://localhost:11434")

vektor = embeddings.embed_query("Der Hund bellt.")
print(len(vektor), "Zahlen, die ersten fünf:", vektor[:5])

### Übung: Ähnlichkeit messen

Die Funktion berechnet, wie ähnlich zwei Texte sind: 1 heißt „gleiche Bedeutung“, Werte nahe 0 heißen „hat nichts miteinander zu tun“.

Probiert eigene Satzpaare aus. Sind Sätze mit ähnlicher Bedeutung, aber anderen Wörtern, ähnlicher als Sätze mit gleichen Wörtern, aber anderer Bedeutung?

In [ ]:
def aehnlichkeit(text_a, text_b):
    a = embeddings.embed_query(text_a)
    b = embeddings.embed_query(text_b)
    skalarprodukt = sum(x * y for x, y in zip(a, b))
    laenge_a = sum(x * x for x in a) ** 0.5
    laenge_b = sum(x * x for x in b) ** 0.5
    return skalarprodukt / (laenge_a * laenge_b)


print(aehnlichkeit("Der Hund bellt.", "Ein Hund macht Lärm."))
print(aehnlichkeit("Der Hund bellt.", "Die Steuererklärung ist fällig."))
# TODO: eigene Satzpaare

# Phase 1: Wissensbasis

Legt eure PDFs in den Ordner `data/` und füllt [quellen.md](quellen.md) und [testfragen.md](testfragen.md) aus.

Nur wenn ihr mit dem Beispiel startet: Entfernt das `#` in der nächsten Zelle, um das Beispiel-PDF nach `data/` zu kopieren.

In [ ]:
# shutil.copy("beispiel/inselhandel-spielregeln.pdf", "data/")
print(sorted(p.name for p in Path("data").glob("*.pdf")))

# Phase 2: Dokumente aufbereiten

### Alle PDFs aus `data/` laden

`PyPDFLoader(pfad).load()` liefert eine Liste mit einem Dokument pro PDF-Seite. Jedes Dokument hat `page_content` (der Text) und `metadata` (Quelle und Seite).

In [ ]:
dokumente = []

for pdf in sorted(Path("data").glob("*.pdf")):
    seiten = PyPDFLoader(str(pdf)).load()
    # TODO: die Seiten zur Liste dokumente hinzufügen (Tipp: dokumente.extend(...))

print(len(dokumente), "PDF-Seiten geladen")

### Website(s) laden

`WebBaseLoader(liste_von_urls).load()` liefert ein Dokument pro Website.

In [ ]:
WEBSITES = [
    # TODO: eure URLs, z. B. "https://de.wikipedia.org/wiki/..."
]

if WEBSITES:
    # TODO: Websites laden und zu dokumente hinzufügen
    pass

print(len(dokumente), "Dokumente insgesamt")

### Kontrolle: Was sieht das LLM eigentlich?

Gebt für jedes Dokument die Quelle und den Anfang des Textes aus. Ist der Text lesbar? Fehlt etwas? Ist Überflüssiges dabei? Ergebnis in [quellen.md](quellen.md) eintragen.

In [ ]:
for doc in dokumente:
    print("=" * 80)
    print(doc.metadata.get("source"), "| Seite:", doc.metadata.get("page", "-"))
    print(doc.page_content[:800])

### Text bereinigen

Die Funktion fasst schon mehrere Leerzeilen zusammen. Ergänzt eigene Regeln, wenn ihr bei der Kontrolle Überflüssiges gefunden habt.

In [ ]:
def bereinigen(text):
    text = re.sub(r"\n\s*\n+", "\n\n", text)  # mehrere Leerzeilen zu einer
    # TODO: weitere Regeln, z. B. Menüeinträge oder Kopfzeilen entfernen
    return text.strip()


for doc in dokumente:
    doc.page_content = bereinigen(doc.page_content)

# Phase 3: Indexieren

### Text in Chunks aufteilen

`chunk_size` ist die maximale Länge eines Chunks in Zeichen, `chunk_overlap` die Überlappung. Die Metadaten werden automatisch in jeden Chunk übernommen.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

chunks = []  # TODO: mit splitter.split_documents(...) aus den Dokumenten Chunks machen

print(len(chunks), "Chunks")

### Vektordatenbank einrichten

Qdrant läuft hier im Arbeitsspeicher. Nach einem Kernel-Neustart ist alles weg, dann die Zellen ab Phase 2 erneut ausführen. Eine bestehende Collection mit gleichem Namen wird vorher gelöscht, damit nichts doppelt drin ist.

In [ ]:
client = QdrantClient(location=":memory:")
SAMMLUNG = "mein_thema"  # TODO: Namen für euer Thema wählen

if client.collection_exists(SAMMLUNG):
    client.delete_collection(SAMMLUNG)

client.create_collection(
    SAMMLUNG,
    # embeddinggemma liefert 768 Zahlen pro Text, verglichen wird mit der Kosinus-Ähnlichkeit
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)
vektorspeicher = QdrantVectorStore(client=client, collection_name=SAMMLUNG, embedding=embeddings)

print([c.name for c in client.get_collections().collections])

### Chunks speichern

`vektorspeicher.add_documents(liste)` berechnet für jeden Chunk das Embedding und speichert es. Das kann einen Moment dauern.

In [ ]:
# TODO: Chunks im Vektorspeicher speichern

print(client.count(SAMMLUNG).count, "Chunks gespeichert")

### Ähnlichkeitssuche ohne LLM

Je höher der Wert, desto ähnlicher ist der Chunk der Frage. Probiert alle eure Testfragen aus: Ist der Chunk mit der Antwort dabei?

In [ ]:
frage = "..."  # TODO: eine eurer Testfragen

for doc, wert in vektorspeicher.similarity_search_with_score(frage, k=4):
    quelle = Path(doc.metadata["source"]).name
    print(f"{wert:.3f} | {quelle} | {doc.page_content[:150]!r}")

# Phase 4: RAG-Abfrage

Die Vorlage für den Prompt ist vorgegeben. `{kontext}` und `{frage}` werden mit `PROMPT.format(kontext=..., frage=...)` ersetzt.

In [ ]:
PROMPT = """Du beantwortest Fragen ausschließlich mit Hilfe des folgenden Kontexts.
Nenne am Ende die Quelle (Datei oder URL und Seite), aus der die Antwort stammt.
Steht die Antwort nicht im Kontext, antworte genau mit: Das steht nicht in meinen Dokumenten.

Kontext:
{kontext}

Frage: {frage}
"""


def fragen(frage, k=4):
    # TODO 1: die k passendsten Chunks holen (vektorspeicher.similarity_search)
    # TODO 2: daraus einen Kontext-Text bauen: pro Chunk eine Zeile mit Quelle und Seite, darunter der Inhalt
    # TODO 3: PROMPT füllen, mit llm.invoke(...) abschicken und den Text der Antwort zurückgeben
    return "TODO"

### Testfragen durchlaufen lassen

In [ ]:
testfragen = [
    # TODO: eure 5 Testfragen aus testfragen.md
]

for frage in testfragen:
    print("FRAGE:", frage)
    print(fragen(frage))
    print("-" * 80)

# Phase 5: Verbessern

Chunk-Größe, Überlappung, `k` und Prompt verändern, immer nur **eine** Sache auf einmal. Danach die Zellen ab „Text in Chunks aufteilen“ erneut ausführen und die Ergebnisse in [testfragen.md](testfragen.md) eintragen.

In [ ]:
# Platz für eure Experimente